In [ ]:
def planner_agent(topic: str, model: str = "gpt-4o-mini") -> list[str]:
    """
    Generates a step-by-step plan for a classification task for a .

    Args:
        topic (str): Research topic or question to investigate.
        model (str): Language model to use.

    Returns:
        list[str]: A valid Python list of executable step strings.
    """
    user_prompt = f"""
You are a planning agent responsible for organizing a hypothesis-testing workflow
for a Type 2 Diabetes Mellitus (T2DM) severity classification project.

Topic:
{topic}

Available agents:
- A data understanding agent that identifies relevant variables in the dataset.
- A model loading agent that loads the trained classifier and preprocessing objects.
- A fake patient agent that creates a baseline synthetic patient from dataset averages or medians.
- A hypothesis application agent that modifies the synthetic patient according to the hypothesis.
- A prediction agent that gets class probabilities from the trained model.
- A comparison agent that compares baseline vs modified predictions.
- A conclusion agent that writes a natural language interpretation of the results.

Your job:
Write a clear, step-by-step plan as a valid Python list, where each step is a short executable string.

Rules:
- Return ONLY a valid Python list.
- Each item must be a string.
- Keep steps atomic and in logical order.
- Do not include explanations outside the list.
- The plan should fit a workflow that tests how a hypothesis changes predicted T2DM severity.

Example style:
[
    "Identify the relevant variables mentioned in the topic.",
    "Load the trained model and preprocessing pipeline.",
    "Create a baseline synthetic patient from dataset averages.",
    "Apply the hypothesis conditions to the synthetic patient.",
    "Generate predictions for the baseline and modified patients.",
    "Compare the class probabilities.",
    "Write a natural language conclusion."
]
"""

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": user_prompt}],
        temperature=0
    )

    plan_text = response.choices[0].message.content.strip()

    try:
        plan = eval(plan_text)
        if isinstance(plan, list) and all(isinstance(step, str) for step in plan):
            return plan
        else:
            raise ValueError("Response was not a valid list of strings.")
    except Exception as e:
        raise ValueError(f"Failed to parse planner output: {e}\nRaw output:\n{plan_text}")

In [ ]:
model = scope["final_mlp"]
preprocessor = scope["preprocessor"]
class_names = scope["class_names"]

In [ ]:
def load_trained_model_agent(topic: str, model: str = "openai:o4-mini") -> list[str]:

    user_prompt = f"""
You are a machine learning planning agent responsible for loading a trained model
and running predictions.

Context:
A trained neural network model and preprocessing pipeline were previously done

The model architecture must be rebuilt using build_mlp before loading weights.

Your job:
Produce a Python list of executable steps describing how to:

1. Load the preprocessing pipeline.
2. Load class names.
3. Rebuild the model architecture using build_mlp.
4. Load the trained model weights.
5. Prepare new input data using the preprocessor.
6. Run model predictions.
7. Convert predicted class indices into class names.
8. Summarize the prediction results.

Rules:
- Each step must be atomic.
- Return ONLY a valid Python list.
- Do not include explanations.

Task: "{topic}"
"""

In [ ]:
import pandas as pd

def create_standard_patient_agent(
    df: pd.DataFrame,
    target_cols: list[str] | None = None
) -> dict:
    """
    Create a standard patient profile for model inference.

    Numeric columns -> median
    Non-numeric columns -> mode
    Excludes target columns.

    Args:
        df (pd.DataFrame): Training dataframe.
        target_cols (list[str] | None): Columns to exclude, such as labels.

    Returns:
        dict: One-row synthetic patient profile.
    """
    if target_cols is None:
        target_cols = []

    feature_df = df.drop(columns=target_cols, errors="ignore")
    patient = {}

    for col in feature_df.columns:
        series = feature_df[col].dropna()

        if len(series) == 0:
            patient[col] = None
        elif pd.api.types.is_numeric_dtype(series):
            patient[col] = float(series.median())
        else:
            mode_vals = series.mode()
            patient[col] = mode_vals.iloc[0] if not mode_vals.empty else None

    return patient

In [ ]:
target_cols = [
    "study_group_healthy",
    "study_group_insulin_dependent",
    "study_group_oral_medication_and_or_non_insulin_injectable_medication_controlled",
    "study_group_pre_diabetes_lifestyle_controlled"
]

standard_patient = create_standard_patient_agent(df)
print(type(standard_patient))
print(list(standard_patient.items())[:10])

In [ ]:
def apply_hypothesis_agent(hypothesis: str) -> dict:
    edits = {}
    parts = hypothesis.split("AND")

    for part in parts:
        part = part.strip()
        if "=" in part:
            feature, level = part.split("=", 1)
            edits[feature.strip()] = level.strip().upper()

    return edits

In [ ]:
def level_to_value(feature: str, level: str, boundaries: dict) -> float:
    """
    Convert LOW / MID / HIGH into a numeric value using feature boundaries.
    """
    if feature not in boundaries:
        raise KeyError(f"Feature '{feature}' not found in BOUNDARIES")

    low, high = boundaries[feature]
    mid = (low + high) / 2

    level = str(level).strip().upper()

    if level == "LOW":
        return low
    elif level in ["MID", "MEDIUM", "NORMAL", "AVERAGE"]:
        return mid
    elif level == "HIGH":
        return high
    else:
        # if already numeric, try to use it directly
        try:
            return float(level)
        except ValueError:
            raise ValueError(
                f"Unsupported level '{level}' for feature '{feature}'. "
                "Use LOW, MID/MEDIUM/NORMAL, HIGH, or a numeric value."
            )

In [ ]:
def apply_changes(patient_dict: dict, edits: dict, boundaries: dict = BOUNDARIES) -> dict:
    """
    Apply hypothesis edits to a patient dictionary.
    
    Example:
    edits = {"Physical Activity": "HIGH", "Heart Rate Pulse": "LOW"}
    """
    updated = patient_dict.copy()

    for feature, level in edits.items():
        if feature in boundaries:
            updated[feature] = level_to_value(feature, level, boundaries)
        else:
            # if the feature is not in boundaries, just write the raw value
            updated[feature] = level

    return updated

In [ ]:
def compare_predictions_agent(topic: str, model: str = "openai:o4-mini") -> str:    
    """
    Compares model predictions before and after applying a hypothesis
    and explains how the predicted T2DM severity changed.

    Args:
        task (str): Description of baseline prediction and modified prediction.
        model (str): Language model to use.

    Returns:
        str: Explanation of prediction differences.
    """

    print("==================================")
    print("📊 Compare Predictions Agent")
    print("==================================")

    system_prompt = """
You are a model interpretation agent for a medical machine learning workflow.

Your job is to compare predictions from a classifier before and after
a hypothesis is applied to a patient profile.

Focus on:
- the predicted class
- prediction probabilities
- confidence changes
- whether severity increased, decreased, or stayed similar

Guidelines:
- Be concise and factual.
- Use the evidence provided in the task.
- Do not invent numbers that are not given.
- Clearly describe the direction of change in prediction.
"""

    response = CLIENT.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
           
            {"role": "user", "content": topic}
        ],
        temperature=1
    )

    return response.choices[0].message.content

In [ ]:
def generate_natural_language_conclusion_agent(task: str, model: str = "openai:o4-mini") -> str:    
    """
    Generates a polished natural-language conclusion for a T2DM hypothesis
    based on model comparison results and supporting evidence.

    Args:
        task (str): Structured evidence about the hypothesis, prediction changes,
                    certainty, and reliability.
        model (str): Language model to use.

    Returns:
        str: Final natural-language conclusion.
    """
    print("==================================")
    print("📝 Natural Language Conclusion Agent")
    print("==================================")

    system_prompt = """
You are a natural language conclusion agent for a medical machine learning workflow.

Your role is to turn structured model findings into a clear, polished final conclusion
about whether a hypothesis meaningfully affects predicted Type 2 Diabetes Mellitus (T2DM)
likelihood or severity.

You must base your conclusion ONLY on the provided evidence.
Do not invent facts, numbers, or effects that are not given.
If evidence is weak or mixed, say so clearly.

Return the answer in exactly this format:

In the OpenAI model,

(<hypothesis>) is <description1> to change the predicted T2DM severity.

The first variable showed:
<brief summary of the first variable's individual effect>

The second variable showed:
<brief summary of the second variable's individual effect>

The combination of the variables showed:
<brief summary of the combined effect>

Therefore, this hypothesis is <description2> to meaningfully affect likelihood of T2DM.

Prediction certainty: <High / Moderate / Low>
Model reliability: <High / Moderate / Low>

Rules:
- Keep the wording cautious, concise, and evidence-based.
- "description1" should be one of:
  likely, somewhat likely, unlikely, not clearly likely, plausibly associated
- "description2" should be one of:
  likely, moderately supported, weakly supported, unlikely, insufficiently supported
- If one variable is clearly stronger than the other, say so.
- If the combination is mostly driven by one variable, say so.
- Do not add markdown headings or bullet points.
"""

    response = CLIENT.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": task}
        ],
        temperature=1
    )

    return response.choices[0].message.content

In [ ]:
import pandas as pd
import numpy as np

def predict_one(patient_dict, model, preprocessor, class_names):
    """
    Predict one patient record.

    Args:
        patient_dict (dict): One patient as a dictionary.
        model: Trained classifier.
        preprocessor: Fitted preprocessing pipeline.
        class_names (list): Class names in model output order.

    Returns:
        dict: Predicted class, confidence, and probabilities.
    """
    # Convert to one-row DataFrame
    X_one = pd.DataFrame([patient_dict])

    # Apply preprocessing
    X_one_p = preprocessor.transform(X_one)

    # Predict probabilities
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_one_p)[0]
    else:
        X_one_p = X_one_p.toarray() if hasattr(X_one_p, "toarray") else X_one_p
        probs = model.predict(X_one_p, verbose=0)[0]
    
    # Top class
    top_idx = int(np.argmax(probs))
    top_class = class_names[top_idx]
    confidence = float(probs[top_idx])

    return {
        "top_class": top_class,
        "confidence": confidence,
        "probs": {class_names[i]: float(probs[i]) for i in range(len(class_names))}
    }

In [ ]:
def run_hypothesis_pipeline(topic, hypothesis):
    print("==================================")
    print("🚀 Starting Hypothesis Pipeline")
    print("==================================")

    print("\n[1] Running planner_agent...")
    plan = planner_agent(topic)
    print("Done.")

    print("\n📋 Plan:")
    for step in plan:
        print("-", step)

    print("\n[2] Running create_standard_patient_agent...")
    standard_patient = create_standard_patient_agent(df, target_cols)
    print("Done.")

    print("\n[3] Running apply_hypothesis_agent...")
    edits = apply_hypothesis_agent(hypothesis)
    print("Done.")

    print("\n[4] Running apply_changes...")
    modified_patient = apply_changes(standard_patient.copy(), edits)
    print("Done.")

    print("\n[5] Running baseline prediction...")
    baseline_pred = predict_one(standard_patient, model, preprocessor, class_names)
    print("Done.")

    print("\n[6] Running changed prediction...")
    changed_pred = predict_one(modified_patient, model, preprocessor, class_names)
    print("Done.")

    print("\n[7] Running compare_predictions_agent...")
    comparison_task = f'''
Baseline prediction:
{baseline_pred}

Prediction after applying hypothesis:
{changed_pred}

Hypothesis:
{hypothesis}

Explain the change in model predictions.
'''
    comparison = compare_predictions_agent(comparison_task)
    print("Done.")

    print("\n[8] Running generate_natural_language_conclusion_agent...")
    final_task = f'''
Hypothesis:
{hypothesis}

Prediction comparison:
{comparison}

Baseline prediction:
{baseline_pred}

Changed prediction:
{changed_pred}
'''
    conclusion = generate_natural_language_conclusion_agent(final_task)
    print("Done.")

    print("\n==================================")
    print("✅ Final Conclusion")
    print("==================================")
    print(conclusion)

    return conclusion

In [ ]:
topic = "Investigate whether physical activity and other variables affect T2DM severity"

pipeline_results = []

for idx, hypothesis in enumerate(question_list, start=1):

    print("\n" + "="*70)
    print(f"Running pipeline for Question {idx}")
    print(hypothesis)
    print("="*70)

    result = run_hypothesis_pipeline(topic, hypothesis)

    pipeline_results.append({
        "question_num": idx,
        "hypothesis": hypothesis,
        "result": result
    })

In [ ]:
pipeline_df = pd.DataFrame(pipeline_results)

pipeline_df.to_csv("pipeline_results_5_questions.csv", index=False)